# Polymarket Arbitrage Bot — v3 Step-by-Step Assembly

This notebook builds the bot incrementally:
1. Authentication — connect to Polymarket CLOB
2. Discovery — find active BTC 5-minute markets
3. Stream — subscribe to live prices via WebSocket
4. Strategy — evaluate arbitrage opportunity (coming next)

## Setup

In [9]:
import sys, os
from pathlib import Path

# VS Code sets __vsc_ipynb_file__ to the notebook's absolute path
try:
    ROOT = str(Path(__vsc_ipynb_file__).parent.parent.parent)
except NameError:
    # Fallback: walk up from cwd until we find src/
    ROOT = next(
        str(p) for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "src").is_dir()
    )

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
print("Root:", ROOT)

Root: /Users/ytwok/dev/project/polymarket-bot


---
## Step 1 — Authentication

`PolymarketClient` wraps `py_clob_client.ClobClient`.

- **L1 auth**: derives API credentials by signing with your private key  
- **L2 auth**: HMAC-SHA256 per-request signing with those credentials  
- `signature_type=1` → POLY_PROXY (exported private key from polymarket.com)

In [10]:
from src.v3.exchange.client import PolymarketClient

pm = PolymarketClient()
clob = pm.client
print("Connected:", pm.ok())

2026-03-11 03:29:25,134 INFO HTTP Request: POST https://clob.polymarket.com/auth/api-key "HTTP/2 400 Bad Request"
2026-03-11 03:29:25,355 INFO HTTP Request: GET https://clob.polymarket.com/auth/derive-api-key "HTTP/2 200 OK"
2026-03-11 03:29:25,660 INFO HTTP Request: GET https://clob.polymarket.com/ "HTTP/2 200 OK"


Connected: OK


In [4]:
# Inspect the underlying ClobClient and resolved contract addresses
clob = pm.client
print("Host         :", clob.host)
print("Chain ID     :", clob.chain_id)

Host         : https://clob.polymarket.com
Chain ID     : 137


---
## Step 2 — Market Discovery

`MarketDiscovery` predicts BTC 5-minute market slugs (`btc-updown-5m-{TS}`) from the current UTC time and fetches them from the Gamma API.

Returns `[current_market, next_market]`, each with YES/NO token IDs and `end_timestamp`.

In [31]:
from src.v3.exchange.discovery import MarketDiscovery
import time

discovery = MarketDiscovery()
markets = discovery.get_current_and_next()

for m in markets:
    closes_in = m['end_timestamp'] - int(time.time())
    print(f"  slug       : {m['slug']}")
    print(f"  closes in  : {closes_in}s")
    print(f"  YES token  : {m['yes']}")
    print(f"  NO  token  : {m['no']}")
    print()

  slug       : btc-updown-5m-1773174600
  closes in  : 299s
  YES token  : 61512226761751245315313771097853697247407431587413148538437098818468014214654
  NO  token  : 22364870954211487988885250769127385530194454357188645088396378641485179046812

  slug       : btc-updown-5m-1773174900
  closes in  : 599s
  YES token  : 33045927086344713822645266436402365569408108841275485288178273037601886454033
  NO  token  : 367164684129975418454445053545229490082296100708864835096577407760603016562



In [33]:
# Quick sanity: fetch orderbook for YES token of current market
if markets:
    book = clob.get_order_book(markets[0]['yes'])
    bids = book.bids[:3] if book.bids else []
    asks = book.asks[:3] if book.asks else []
    print("Top bids:", [(b.price, b.size) for b in bids])
    print("Top asks:", [(a.price, a.size) for a in asks])

2026-03-11 03:30:13,559 INFO HTTP Request: GET https://clob.polymarket.com/book?token_id=61512226761751245315313771097853697247407431587413148538437098818468014214654 "HTTP/2 200 OK"


Top bids: [('0.01', '9175.34'), ('0.02', '6741.35'), ('0.03', '3031.32')]
Top asks: [('0.99', '9531.24'), ('0.98', '6806.35'), ('0.97', '3058.59')]


---
## Step 3 — WebSocket Price Streaming

`MarketStreamer` manages the full streaming lifecycle:
- Subscribes to YES + NO tokens for current and next market
- Sends `PING` heartbeat every 10s
- Auto-rotates to the next market when `now >= end_timestamp`
- Calls `async callback(market, prices)` on every `best_bid_ask` event

We'll run it for **30 seconds** to observe live price updates.

In [34]:
import asyncio
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

from src.v3.exchange.stream import MarketStreamer

streamer = MarketStreamer(discovery)

async def on_price(market, prices):
    yes_ask = prices.get(market['yes'], {}).get('ask')
    no_ask  = prices.get(market['no'],  {}).get('ask')
    if yes_ask and no_ask:
        total = yes_ask + no_ask
        if total < 1.0:
            print(f"[{market['slug']}]  YES ask={yes_ask:.3f}  NO ask={no_ask:.3f}  total={total:.3f}")

async def run_for(seconds: int):
    task = asyncio.create_task(streamer.run(on_price))
    await asyncio.sleep(seconds)
    task.cancel()
    try:
        await task
    except asyncio.CancelledError:
        print("\nStream stopped.")

await run_for(30)

2026-03-11 03:31:29,200 INFO [CURRENT] btc-updown-5m-1773174600  closes=1773174900
2026-03-11 03:31:29,201 INFO [NEXT] btc-updown-5m-1773174900  closes=1773175200
2026-03-11 03:31:29,702 INFO WebSocket connected.
2026-03-11 03:31:29,703 INFO Subscribed to 4 tokens.



Stream stopped.


---
## Step 4 — What's next?

With live YES/NO ask prices flowing, the next modules to build are:

| Module | File | Role |
|--------|------|------|
| **Evaluator** | `trading/evaluator.py` | Is `YES_ask + NO_ask + fees < $1.00`? |
| **Executor** | `trading/executor.py` | Place FOK orders on both sides |
| **Merger** | `trading/merger.py` | Call `CTF.mergePositions()` to recover USDC |

Run the full bot:
```bash
python -m src.v3.main
```